# E4 — cgroup Memory Cap + OOM Kill Analysis

**Experiment:** E4-mem-cap  
**Layer:** L4 — compose memory limit (`deploy.resources.limits.memory`)  
**Scenarios:** S0-baseline vs S8-mem-cap-512m (`BACK_MEM=512m`)  
**Workload:** W2-upload at medium intensity

## Research question

When the cgroup memory limit (512m) is smaller than the JVM's own `-Xmx` (2g):  
1. Does `container_memory_failcnt > 0` fire as the OOM fingerprint at the **container** layer?
2. Are the JVM heap metrics **healthy** at the time of kill?  
   (This is the key that distinguishes S8 from S1 — same upload workload, different root cause.)
3. Does the recommender correctly propose `BACK_MEM=2g` (compose limit) instead of `-Xmx` increase?

## S8 vs S1: the vertical-layer confusion trap

| Signal | S1 heap-256m | S8 mem-cap-512m |
|--------|--------------|------------------|
| Workload | W2-upload | W2-upload |
| User symptom | upload failures, high p99 | upload truncated, container restarts |
| `heap_ratio` | > 0.90 (JVM heap exhausted) | < 0.50 (JVM heap healthy) |
| `container_memory_failcnt` | 0 (no cgroup OOM) | > 0 (cgroup OOM fires) |
| `container_restarts` | 0 | > 0 |
| **Root cause** | JVM `-Xmx` too small | compose `mem_limit` too small |
| **Fix** | `JAVA_TOOL_OPTIONS=-Xmx2g` | `BACK_MEM=2g` |

## Figures produced
- **Fig 1** — OOM events: `container_memory_failcnt` and restart count time series
- **Fig 2** — JVM heap stays healthy (proving OOM is at cgroup layer, not JVM)
- **Fig 3** — S8 vs S1 disambiguation panel (side-by-side diagnostic signals)

Run `python experiments/E4-mem-cap/run.py` first.

In [ ]:
import json
import csv
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import yaml
import pandas as pd
from IPython.display import display

matplotlib.rcParams.update({
    'font.size': 11, 'axes.titlesize': 12, 'axes.labelsize': 11,
    'legend.fontsize': 10, 'figure.dpi': 150,
    'savefig.dpi': 300, 'savefig.bbox': 'tight',
})

REPO_ROOT = Path('.').resolve().parent.parent
RESULTS_DIR = REPO_ROOT / 'results'
FIGURES_DIR = Path('.') / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)

E4_KEY = 'E4-mem-cap'
E1_KEY = 'E1-jvm-heap'   # loaded for the S8-vs-S1 comparison figure

COLORS = {
    'S0-baseline':    '#2196F3',
    'S8-mem-cap-512m':'#F44336',
    'S1-heap-256m':   '#FF9800',
}
LABELS = {
    'S0-baseline':    'S0 baseline',
    'S8-mem-cap-512m':'S8 mem-cap-512m (cgroup OOM)',
    'S1-heap-256m':   'S1 heap-256m (JVM OOM)',
}

In [ ]:
def load_locust_stats(run_dir: Path) -> dict:
    f = run_dir / 'locust_stats.csv'
    if not f.exists():
        return {}
    rows = {}
    with open(f) as fh:
        for row in csv.DictReader(fh):
            name = row.get('Name', '')
            try:
                total = float(row.get('Request Count', 1) or 1)
                rows[name] = {
                    'p99':        float(row.get('99%', 0) or 0),
                    'error_rate': float(row.get('Failure Count', 0) or 0) / max(1, total),
                    'rps':        float(row.get('Requests/s', 0) or 0),
                }
            except (ValueError, TypeError):
                pass
    return rows


def get_prom_series(prom: dict, key: str):
    data = prom.get(key, {})
    if data.get('status') != 'success':
        return [], []
    ts_list, v_list = [], []
    for series in data.get('data', {}).get('result', []):
        for ts, v in series.get('values', []):
            try:
                ts_list.append(float(ts))
                v_list.append(float(v))
            except (ValueError, TypeError):
                pass
    return ts_list, v_list


def load_run(run_id: str, experiment_key: str) -> dict:
    d = RESULTS_DIR / run_id
    meta = yaml.safe_load((d / 'metadata.yaml').read_text()) if (d / 'metadata.yaml').exists() else {}
    prom = json.loads((d / 'prom_snapshot.json').read_text()) if (d / 'prom_snapshot.json').exists() else {}
    meta.setdefault('experiment', experiment_key)
    return {'run_id': run_id, 'meta': meta, 'prom': prom, 'stats': load_locust_stats(d)}

In [ ]:
runs_e4, runs_e1 = [], []
if RESULTS_DIR.exists():
    for run_dir in sorted(RESULTS_DIR.iterdir()):
        mf = run_dir / 'metadata.yaml'
        if not mf.exists():
            continue
        meta = yaml.safe_load(mf.read_text())
        exp = meta.get('experiment', '')
        if exp == E4_KEY:
            runs_e4.append(load_run(run_dir.name, E4_KEY))
        elif exp == E1_KEY:
            runs_e1.append(load_run(run_dir.name, E1_KEY))

print(f'E4-mem-cap:   {len(runs_e4)} run(s)')
print(f'E1-jvm-heap:  {len(runs_e1)} run(s)  (used for S8 vs S1 comparison)')
for r in runs_e4 + runs_e1:
    m = r['meta']
    print(f"  {r['run_id'][:8]}  exp={m.get('experiment','?')[:12]:12s}  "
          f"scenario={m.get('scenario','?')}  intensity={m.get('intensity','?')}")

if not runs_e4:
    print('\nNo E4 results yet. Generate data with:')
    print('  python experiments/E4-mem-cap/run.py')

In [ ]:
records = []
for r in runs_e4:
    m, prom, stats = r['meta'], r['prom'], r['stats']
    agg = stats.get('Aggregated', next(iter(stats.values()), {}))

    key_fail = "container_memory_failcnt{name='correctexam-back'}"
    key_rst  = "container_restarts{name='correctexam-back'}"
    key_mem  = "container_memory_usage_bytes{name='correctexam-back'}"

    _, failcnt  = get_prom_series(prom, key_fail)
    _, restarts = get_prom_series(prom, key_rst)
    _, mem      = get_prom_series(prom, key_mem)

    _, heap_used = get_prom_series(prom, "jvm_memory_used_bytes{area='heap'}")
    _, heap_max  = get_prom_series(prom, "jvm_memory_max_bytes{area='heap'}")
    ratios = [u / mx for u, mx in zip(heap_used, heap_max) if mx > 0]

    records.append({
        'scenario':        m.get('scenario', '?'),
        'intensity':       m.get('intensity', '?'),
        'run_id':          r['run_id'][:8],
        'p99_ms':          agg.get('p99', 0),
        'error_rate_%':    round(agg.get('error_rate', 0) * 100, 2),
        'failcnt_max':     max(failcnt)  if failcnt  else 0,
        'restarts_max':    max(restarts) if restarts else 0,
        'mem_max_mb':      round(max(mem) / 1e6, 1) if mem else 0,
        'heap_ratio_max':  round(max(ratios), 3) if ratios else 0,
    })

df = pd.DataFrame(records)
if not df.empty:
    display(df)
else:
    print('No data yet.')

In [ ]:
# Figure 1 — OOM events: memory failcnt and container restarts (S0 vs S8)

target = {'S0-baseline': None, 'S8-mem-cap-512m': None}
for r in runs_e4:
    sc = r['meta'].get('scenario')
    if sc in target:
        target[sc] = r

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Fig 1 — E4 OOM Kill Signal: container_memory_failcnt and Restarts (W2-upload, medium)',
             fontsize=12, fontweight='bold')

for scenario, r in target.items():
    c   = COLORS[scenario]
    lbl = LABELS[scenario]

    if r is None:
        for ax in axes:
            ax.text(0.5, 0.5, f'Run for {scenario}\nnot found',
                    transform=ax.transAxes, ha='center', va='center',
                    fontsize=10, color='gray', style='italic')
        continue

    prom = r['prom']
    ts_f, failcnt  = get_prom_series(prom, "container_memory_failcnt{name='correctexam-back'}")
    ts_r, restarts = get_prom_series(prom, "container_restarts{name='correctexam-back'}")

    if ts_f and failcnt:
        t_rel = [(t - ts_f[0]) / 60 for t in ts_f]
        axes[0].plot(t_rel, failcnt, color=c, label=lbl, linewidth=2)

    if ts_r and restarts:
        t_rel = [(t - ts_r[0]) / 60 for t in ts_r]
        axes[1].plot(t_rel, restarts, color=c, label=lbl, linewidth=2)

axes[0].set_title('Memory Failcnt (cgroup OOM events)')
axes[0].set_xlabel('Time into run (min)')
axes[0].set_ylabel('container_memory_failcnt')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Container Restarts')
axes[1].set_xlabel('Time into run (min)')
axes[1].set_ylabel('container_restarts')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'E4-fig1-oom-signal.pdf')
plt.show()
print('Saved: figures/E4-fig1-oom-signal.pdf')

In [ ]:
# Figure 2 — JVM heap stays healthy during S8 (heap_ratio stays low)
# This is the key signal that proves the OOM is at the CONTAINER layer, not JVM.

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Fig 2 — E4 JVM Heap Healthy at OOM: Proving the Kill is at the cgroup Layer',
             fontsize=12, fontweight='bold')

for scenario, r in target.items():
    c   = COLORS[scenario]
    lbl = LABELS[scenario]

    if r is None:
        continue

    prom = r['prom']
    ts_u, heap_used = get_prom_series(prom, "jvm_memory_used_bytes{area='heap'}")
    _,    heap_max  = get_prom_series(prom, "jvm_memory_max_bytes{area='heap'}")
    ts_m, mem_bytes = get_prom_series(prom, "container_memory_usage_bytes{name='correctexam-back'}")

    if ts_u and heap_used and heap_max:
        ratios = [u / mx if mx > 0 else 0 for u, mx in zip(heap_used, heap_max)]
        t_rel  = [(t - ts_u[0]) / 60 for t in ts_u]
        axes[0].plot(t_rel, ratios, color=c, label=lbl, linewidth=2)

    if ts_m and mem_bytes:
        mem_mb = [b / 1e6 for b in mem_bytes]
        t_rel  = [(t - ts_m[0]) / 60 for t in ts_m]
        axes[1].plot(t_rel, mem_mb, color=c, label=lbl, linewidth=2)

axes[0].axhline(0.90, color='orange', linestyle='--', linewidth=1.2, alpha=0.8,
                label='S1 diagnostic threshold (0.90)')
axes[0].set_title('JVM Heap Ratio (used / max)')
axes[0].set_xlabel('Time into run (min)')
axes[0].set_ylabel('heap_used / heap_max')
axes[0].set_ylim(0, 1.1)
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].text(0.02, 0.97, 'Key: S8 heap_ratio < 0.90 → OOM is NOT a JVM heap issue',
             transform=axes[0].transAxes, fontsize=9, va='top',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

axes[1].axhline(512, color='red', linestyle='--', linewidth=1.2, alpha=0.8,
                label='cgroup limit: 512 MB')
axes[1].set_title('Container Memory Usage (MB)')
axes[1].set_xlabel('Time into run (min)')
axes[1].set_ylabel('memory usage (MB)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'E4-fig2-heap-healthy.pdf')
plt.show()
print('Saved: figures/E4-fig2-heap-healthy.pdf')

In [ ]:
# Figure 3 — S8 vs S1 disambiguation: same workload, different diagnostic signal
# Left: S1 heap-256m (JVM heap ratio > 0.90, failcnt = 0)
# Right: S8 mem-cap-512m (heap ratio OK, failcnt > 0)

s1_run = next((r for r in runs_e1
               if r['meta'].get('scenario') == 'S1-heap-256m'
               and r['meta'].get('intensity') == 'medium'), None)
s8_run = target.get('S8-mem-cap-512m')

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle(
    'Fig 3 — S8 vs S1 Disambiguation: Same Workload (W2-upload), Different Layer\n'
    'Left: JVM OOM (S1)  |  Right: cgroup OOM (S8)',
    fontsize=12, fontweight='bold'
)

run_pairs = [('S1-heap-256m', s1_run, 0), ('S8-mem-cap-512m', s8_run, 1)]

for scenario, r, col in run_pairs:
    c = COLORS[scenario]

    if r is None:
        axes[0, col].text(0.5, 0.5, f'{scenario}\nnot found',
                          transform=axes[0, col].transAxes, ha='center', va='center',
                          fontsize=10, color='gray', style='italic')
        axes[1, col].text(0.5, 0.5, f'{scenario}\nnot found',
                          transform=axes[1, col].transAxes, ha='center', va='center',
                          fontsize=10, color='gray', style='italic')
        continue

    prom = r['prom']

    # Row 0: JVM heap ratio
    ts_u, hu = get_prom_series(prom, "jvm_memory_used_bytes{area='heap'}")
    _, hm    = get_prom_series(prom, "jvm_memory_max_bytes{area='heap'}")
    if ts_u and hu and hm:
        ratios = [u / mx if mx > 0 else 0 for u, mx in zip(hu, hm)]
        t_rel  = [(t - ts_u[0]) / 60 for t in ts_u]
        axes[0, col].plot(t_rel, ratios, color=c, linewidth=2, label='heap_ratio')
    axes[0, col].axhline(0.90, color='orange', linestyle='--', alpha=0.8, label='threshold (0.90)')
    axes[0, col].set_title(f'{scenario}\nJVM Heap Ratio')
    axes[0, col].set_ylabel('heap_used / heap_max')
    axes[0, col].set_ylim(0, 1.15)
    axes[0, col].legend(fontsize=9)
    axes[0, col].grid(True, alpha=0.3)

    # Row 1: container_memory_failcnt
    ts_f, fc = get_prom_series(prom, "container_memory_failcnt{name='correctexam-back'}")
    if ts_f and fc:
        t_rel = [(t - ts_f[0]) / 60 for t in ts_f]
        axes[1, col].plot(t_rel, fc, color=c, linewidth=2, label='memory_failcnt')
    axes[1, col].axhline(0, color='green', linestyle='--', alpha=0.6, label='threshold (0 = no OOM)')
    axes[1, col].set_title('container_memory_failcnt')
    axes[1, col].set_xlabel('Time into run (min)')
    axes[1, col].set_ylabel('failcnt')
    axes[1, col].legend(fontsize=9)
    axes[1, col].grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'E4-fig3-s8-vs-s1.pdf')
plt.show()
print('Saved: figures/E4-fig3-s8-vs-s1.pdf')

## Interpretation

### Diagnostic rule (from diagnose.py)
```
IF container_memory_failcnt > 0  OR  container_restarts > 0 (during run)
AND jvm_heap_ratio < 0.90  (JVM heap is healthy — NOT S1)
THEN scenario = S8-mem-cap-512m, layer = L4, confidence = 0.9
```

### Why the heap check matters
Without checking `jvm_heap_ratio`, both S1 and S8 could trigger on the same symptom  
(upload failures, container instability). The heap ratio is the discriminator:
- S1: JVM ran out of heap → `heap_ratio > 0.90` → fix is `-Xmx2g`
- S8: cgroup killed the container before JVM hit its Xmx → `heap_ratio < 0.50` → fix is `BACK_MEM=2g`

### Expected outcome
| Metric | S0 | S8-mem-cap-512m |
|--------|----|-----------------|
| `container_memory_failcnt` | 0 | > 0 |
| `container_restarts` | 0 | ≥ 1 |
| `heap_ratio_max` | < 0.60 | < 0.60 (stays healthy!) |
| `error_rate_%` | < 1% | > 5% (uploads truncated at restart) |